# Figure 2b-d codes

This notebook contains the plotting code for main-text Figure 2b, Figure 2c, and Figure 2d using the CSV files in `data/`.


In [ ]:
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
import shutil

BASE_DIR = Path.cwd()
DATA_DIR = BASE_DIR / 'data'
OUTPUT_DIR = BASE_DIR / 'output'
OUTPUT_DIR.mkdir(exist_ok=True)

plt.rcParams['font.family'] = 'Arial'
plt.rcParams['font.size'] = 25

COLORS = sns.color_palette('vlag', 5)
BARCOLORS_2C = ['#5EA5C5', '#F08743', '#559F59']
TOPIC_ORDER_2C = ['Politics', 'GunControl', 'AbortionBan']
TOPIC_LABELS_2C = ['Partisan\nAlignment', 'Gun\nControl', 'Abortion\nBan']
TOPIC_ORDER_2D = ['Politics_de', 'Gun_Control_de', 'Abortion_de']
TITLE_BY_TOPIC_2D = {'Politics_de': 'Partisan Alignment', 'Gun_Control_de': 'Gun Control', 'Abortion_de': 'Abortion Ban'}


## Figure 2b


In [ ]:
pairwise = pd.read_csv(DATA_DIR / 'figure2b_pairwise_abortionban.csv')
pairwise_matrix = (
    pairwise.pivot(index='source_group_order', columns='target_state_order', values='proportion')
    .sort_index()
    .to_numpy()
)

fig, axs = plt.subplots(1, pairwise_matrix.shape[0], figsize=(17, 3))
for j, ax in enumerate(axs):
    ax.bar(np.arange(pairwise_matrix.shape[1]), pairwise_matrix[j], color=COLORS, edgecolor="k", lw=1)
    ax.set_ylim(0, 1)
    ax.set_xticks(np.arange(5))
    ax.set_xticklabels('')
axs[0].set_ylabel('Proportion')
plt.tight_layout()
fig.savefig(OUTPUT_DIR / 'figure2b_bias_pairwise.pdf', bbox_inches='tight', pad_inches=0.1, dpi=300)
plt.show()


## Figure 2c


In [ ]:
bias = pd.read_csv(DATA_DIR / 'figure2c_self_inconsistency.csv')
fig, ax = plt.subplots(1, 1, figsize=(15, 5))
idx = [0, 1, 2]
topics_name_shows = np.array(['Partisan\nAlignment', 'Gun\nControl', 'Abortion\nBan'])
barcolors = ['#5EA5C5', '#F08743', '#559F59']
for i, topic in enumerate(TOPIC_ORDER_2C):
    values = bias[bias['topic'] == topic].set_index('condition')['self_inconsistency_rate']
    original = values['original']
    regulated = values['self_regulated']
    ax.bar(i - 0.2, original, width=0.25, color=barcolors[i])
    ax.bar(i + 0.2, regulated, width=0.25, color=barcolors[i], alpha=0.6)
    ax.plot([i - 0.2, i + 0.2], [original, regulated], lw=2, ls='--', color='gray', zorder=1)
    ax.scatter(i - 0.2, original, marker='^', s=100, color=barcolors[i], edgecolors='k')
    ax.scatter(i + 0.2, regulated, marker='o', s=100, color=barcolors[i], edgecolors='k')

ax.set_xticks(np.arange(3))
ax.set_xticklabels(topics_name_shows[idx])
ax.set_ylabel('Self-inconsistency Rate')
ax.set_ylim(0, 0.8)
fig.savefig(OUTPUT_DIR / 'figure2c_self_inconsistency.pdf', bbox_inches='tight', pad_inches=0.1, dpi=300)
plt.show()


## Figure 2d


In [ ]:
node_states = pd.read_csv(DATA_DIR / 'figure2d_debiased_node_state_proportions.csv')

def state_matrix(topic):
    topic_df = node_states[node_states['topic'] == topic]
    matrix = (topic_df.pivot(index='state_order', columns='epoch', values='proportion')
        .sort_index()
        .to_numpy())
    return matrix

for topic in TOPIC_ORDER_2D:
    matrix = state_matrix(topic)
    assert matrix.shape == (5, 41)
    np.testing.assert_allclose(matrix.sum(axis=0), 1.0)

fig, axes = plt.subplots(1, 3, figsize=(17, 5))
for i, topic in enumerate(TOPIC_ORDER_2D):
    data_matrix = np.cumsum(state_matrix(topic), axis=0)
    data_matrix = np.concatenate([np.zeros((1, data_matrix.shape[1])), data_matrix], axis=0)
    for state_idx in range(1, 6):
        line_pre = data_matrix[state_idx - 1, :]
        line_after = data_matrix[state_idx, :]
        axes[i].fill_between(np.arange(line_pre.shape[0]), line_pre, line_after, color=COLORS[state_idx - 1])
    axes[i].set_ylim(0, 1)
    axes[i].set_xlim(0, line_pre.shape[0] - 1)
    axes[i].set_xticks(range(0, line_pre.shape[0], 10))
    axes[i].set_xlabel('Time')
    axes[i].set_title(TITLE_BY_TOPIC_2D[topic], fontsize=25, fontweight="bold")
    if i == 0:
        axes[i].set_ylabel('Proportion')
fig.savefig(OUTPUT_DIR / 'figure2d_opinion_dynamics.pdf', bbox_inches='tight', pad_inches=0.1, dpi=300)
plt.show()
